# Model Evaluation
We will be testing three different models (XGBoost, Random Forest, and Temporal CNN) using the Edge IoT Predictive Irrigation Dataset from Mendeley to see which one performs the best. 

In [1]:
import pandas as pd
import numpy as np

Read the dataset and show the first few columns.

In [2]:
df = pd.read_excel('./Edge_IoT_Predictive_Irrigation_Dataset.xlsx')
df.head()

,Timestamp,Day,Time,Temperature_C,Humidity_pct,Soil_Moisture_pct,Light_Intensity,Pump_Status
0,2026-04-01 00:00,1,00:00,25.1,83,64.1,0,0
1,2026-04-01 00:30,1,00:30,25.0,84,63.4,0,0
2,2026-04-01 01:00,1,01:00,25.0,86,63.4,0,0
3,2026-04-01 01:30,1,01:30,25.0,87,62.8,0,0
4,2026-04-01 02:00,1,02:00,25.0,85,63.1,0,0


In [3]:
df.tail()

,Timestamp,Day,Time,Temperature_C,Humidity_pct,Soil_Moisture_pct,Light_Intensity,Pump_Status
955,2026-04-20 21:30,20,21:30,26.2,81,59.9,0,0
956,2026-04-20 22:00,20,22:00,27.2,79,59.9,0,0
957,2026-04-20 22:30,20,22:30,26.4,83,59.0,0,0
958,2026-04-20 23:00,20,23:00,26.1,82,59.1,0,0
959,2026-04-20 23:30,20,23:30,25.5,85,58.6,0,0


The Timestamp column needs to be converted into the DateTime data type. We use pd.to_datetime() to do this. We then set the index to the Timestamp column to make it a time series dataset.

In [4]:
df["Timestamp"] = pd.to_datetime(df["Timestamp"], errors="coerce")
df = df.set_index("Timestamp")
df.index    # Verify that the index is in DateTime format

DatetimeIndex(['2026-04-01 00:00:00', '2026-04-01 00:30:00',
               '2026-04-01 01:00:00', '2026-04-01 01:30:00',
               '2026-04-01 02:00:00', '2026-04-01 02:30:00',
               '2026-04-01 03:00:00', '2026-04-01 03:30:00',
               '2026-04-01 04:00:00', '2026-04-01 04:30:00',
               ...
               '2026-04-20 19:00:00', '2026-04-20 19:30:00',
               '2026-04-20 20:00:00', '2026-04-20 20:30:00',
               '2026-04-20 21:00:00', '2026-04-20 21:30:00',
               '2026-04-20 22:00:00', '2026-04-20 22:30:00',
               '2026-04-20 23:00:00', '2026-04-20 23:30:00'],
              dtype='datetime64[ns]', name='Timestamp', length=960, freq=None)

We need to check for null values first.

In [5]:
df.isnull().sum()

Day                  0
Time                 0
Temperature_C        0
Humidity_pct         0
Soil_Moisture_pct    0
Light_Intensity      0
Pump_Status          0
dtype: int64

Display the summary statistics for the data

In [6]:
display(df.describe())

,Day,Temperature_C,Humidity_pct,Soil_Moisture_pct,Light_Intensity,Pump_Status
count,960.000000,960.000000,960.000000,960.000000,960.000000,960.000000
mean,10.500000,29.160625,74.484375,65.772500,287.389583,0.029167
std,5.769287,3.389549,8.563190,9.154894,347.983706,0.168361
min,1.000000,25.000000,61.000000,46.600000,0.000000,0.000000
25%,5.750000,25.500000,66.000000,59.600000,0.000000,0.000000
50%,10.500000,29.000000,75.000000,63.600000,25.000000,0.000000
75%,15.250000,32.525000,83.000000,68.325000,640.250000,0.000000
max,20.000000,35.000000,88.000000,85.000000,949.000000,1.000000


We need to create additional features for each part of the Timestamp (Day is already provided). I have excluded milliseconds since it is 00 for all of the values. After the features are created, we will drop time because it is redundant.

In [7]:
def make_baseline_features(df):
    df["Year"] = df.index.year
    df["Month"] = df.index.month
    df["Hour"] = df.index.hour
    df["Minute"] = df.index.minute
    df.drop("Time", axis=1, inplace=True)
    return df

df = make_baseline_features(df)
df

,Day,Temperature_C,Humidity_pct,Soil_Moisture_pct,Light_Intensity,Pump_Status,Year,Month,Hour,Minute
Timestamp,,,,,,,,,,
2026-04-01 00:00:00,1,25.1,83,64.1,0,0,2026,4,0,0
2026-04-01 00:30:00,1,25.0,84,63.4,0,0,2026,4,0,30
2026-04-01 01:00:00,1,25.0,86,63.4,0,0,2026,4,1,0
2026-04-01 01:30:00,1,25.0,87,62.8,0,0,2026,4,1,30
2026-04-01 02:00:00,1,25.0,85,63.1,0,0,2026,4,2,0
...,...,...,...,...,...,...,...,...,...,...
2026-04-20 21:30:00,20,26.2,81,59.9,0,0,2026,4,21,30
2026-04-20 22:00:00,20,27.2,79,59.9,0,0,2026,4,22,0
2026-04-20 22:30:00,20,26.4,83,59.0,0,0,2026,4,22,30


We need to calculate the changes in temperature, humidity, soil moisture, light intensity, and pump status over the previous 30 minutes.

In [8]:
df["Prev_30min_Temperature"] = df["Temperature_C"].shift(1).fillna(0)
df["Prev_30min_Humidity"] = df["Humidity_pct"].shift(1).fillna(0)
df["Prev_30min_Soil_Moisture"] = df["Soil_Moisture_pct"].shift(1).fillna(0)
df["Prev_30min_Light_Intensity"] = df["Light_Intensity"].shift(1).fillna(0)
df["Prev_30min_Pump_Status"] = df["Pump_Status"].shift(1).fillna(0)

In [9]:
df

,Day,Temperature_C,Humidity_pct,Soil_Moisture_pct,Light_Intensity,Pump_Status,Year,Month,Hour,Minute,Prev_30min_Temperature,Prev_30min_Humidity,Prev_30min_Soil_Moisture,Prev_30min_Light_Intensity,Prev_30min_Pump_Status
Timestamp,,,,,,,,,,,,,,,
2026-04-01 00:00:00,1,25.1,83,64.1,0,0,2026,4,0,0,0.0,0.0,0.0,0.0,0.0
2026-04-01 00:30:00,1,25.0,84,63.4,0,0,2026,4,0,30,25.1,83.0,64.1,0.0,0.0
2026-04-01 01:00:00,1,25.0,86,63.4,0,0,2026,4,1,0,25.0,84.0,63.4,0.0,0.0
2026-04-01 01:30:00,1,25.0,87,62.8,0,0,2026,4,1,30,25.0,86.0,63.4,0.0,0.0
2026-04-01 02:00:00,1,25.0,85,63.1,0,0,2026,4,2,0,25.0,87.0,62.8,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-04-20 21:30:00,20,26.2,81,59.9,0,0,2026,4,21,30,26.8,77.0,59.1,0.0,0.0
2026-04-20 22:00:00,20,27.2,79,59.9,0,0,2026,4,22,0,26.2,81.0,59.9,0.0,0.0
2026-04-20 22:30:00,20,26.4,83,59.0,0,0,2026,4,22,30,27.2,79.0,59.9,0.0,0.0


## XGBoost

In [11]:
import xgboost
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

X = df.drop('Pump_Status', axis=1)
y = df['Pump_Status']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=101, stratify=y)
neg, pos = (y_train == 0).sum(), (y_train == 1).sum()
params = {
    'objective':'binary:logistic',
    'max_depth':4,
    'learning_rate':0.1,
    'n_estimators':100,
    'alpha':10,
    'scale_pos_weight': neg/pos
}
xgb_model = XGBClassifier(**params)
xgb_model.fit(X_train, y_train)
y_pred = xgb_model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
print(f"Model Accuracy: {accuracy}")
print("\nConfusion Matrix")
print(confusion_matrix(y_test, y_pred))
print("\nClassification Report")
print(classification_report(y_test, y_pred))

Model Accuracy: 0.9930555555555556

Confusion Matrix
[[279   1]
 [  1   7]]

Classification Report
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       280
           1       0.88      0.88      0.88         8

    accuracy                           0.99       288
   macro avg       0.94      0.94      0.94       288
weighted avg       0.99      0.99      0.99       288



## Random Forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=101, stratify=y)
rf_model = RandomForestClassifier(n_estimators = 100, random_state=101)
rf_model.fit(X_train, y_train)
y_pred = rf_model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f"Model Accuracy: {accuracy}")
print("\nConfusion Matrix")
print(confusion_matrix(y_test, y_pred))
print("\nClassification Report")
print(classification_report(y_test, y_pred))

Model Accuracy: 0.9930555555555556

Classification Report
              precision    recall  f1-score   support

           0       0.99      1.00      1.00       280
           1       1.00      0.75      0.86         8

    accuracy                           0.99       288
   macro avg       1.00      0.88      0.93       288
weighted avg       0.99      0.99      0.99       288



## Temporal CNN

In [61]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Flatten, Dense, Dropout
from sklearn.metrics import classification_report, accuracy_score
from sklearn.preprocessing import StandardScaler

features = ['Temperature_C', 'Humidity_pct', 'Soil_Moisture_pct', 'Light_Intensity']
window = 24
data = df[features].values
labels = df['Pump_Status'].values
X_seq, y_seq = [], []
for i in range(window, len(df)):
    X_seq.append(data[i-window:i])
    y_seq.append(labels[i])

X_seq = np.array(X_seq)
y_seq = np.array(y_seq)

split_idx = int(len(X_seq) * 0.7)
X_train, X_test = X_seq[:split_idx], X_seq[split_idx:]
y_train, y_test = y_seq[:split_idx], y_seq[split_idx:]

n_features = len(features)
scaler = StandardScaler()
X_train_2d = X_train.reshape(-1, n_features)
X_test_2d = X_test.reshape(-1, n_features)

scaler.fit(X_train_2d)
X_train = scaler.transform(X_train_2d).reshape(X_train.shape)
X_test = scaler.transform(X_test_2d).reshape(X_test.shape)

neg, pos = (y_train == 0).sum(), (y_train == 1).sum()
class_weight = {0: 1.0, 1: neg/pos}

cnn_model = Sequential([
    Conv1D(filters=32, kernel_size=10, activation='relu', input_shape=(window, n_features)),
    MaxPooling1D(pool_size=2),
    Conv1D(filters=16, kernel_size=7, activation='relu'),
    Flatten(),
    Dense(16, activation='relu'),
    Dropout(0.3),
    Dense(1, activation='sigmoid')
])

cnn_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

history = cnn_model.fit(
    X_train, y_train,
    validation_split=0.2,
    epochs=50,
    batch_size=16,
    class_weight=class_weight,
    verbose=1
)

y_pred_prob = cnn_model.predict(X_test).flatten()
y_pred = (y_pred_prob > 0.5).astype(int)
accuracy = accuracy_score(y_test, y_pred)
print(f"Model accuracy: {accuracy}")
print("\nClassification Report")
print(classification_report(y_test, y_pred))



c:\Users\drewu\anaconda3\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/50
33/33 ━━━━━━━━━━━━━━━━━━━━ 3s 17ms/step - accuracy: 0.6775 - loss: 1.4030 - val_accuracy: 0.7481 - val_loss: 0.6376
Epoch 2/50
33/33 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.7405 - loss: 1.0866 - val_accuracy: 0.8244 - val_loss: 0.5372
Epoch 3/50
33/33 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.8149 - loss: 0.9315 - val_accuracy: 0.8931 - val_loss: 0.3639
Epoch 4/50
33/33 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.8760 - loss: 0.8474 - val_accuracy: 0.8244 - val_loss: 0.4493
Epoch 5/50
33/33 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.8378 - loss: 0.8043 - val_accuracy: 0.9008 - val_loss: 0.2867
Epoch 6/50
33/33 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.8855 - loss: 0.7015 - val_accuracy: 0.9160 - val_loss: 0.2497
Epoch 7/50
33/33 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.9008 - loss: 0.6401 - val_accuracy: 0.9313 - val_loss: 0.2053
Epoch 8/50
33/33 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.9275 - loss: 0.6029 - val_accuracy: 0.8779 - val_los

Based on our findings, we will use XGBoost because it is tied for highest accuracy with Random Forest, but it does slightly better in evaluating the use case when pump status is 1.